In [31]:
import sys
sys.path.append('../src/')
sys.path.append('../../AIRE/python')
sys.path.append('../')
import os
import copy
from typing import List, Union

import numpy as np
import tensorly as tn
import torch
import math
import matlab.engine
import matplotlib.pyplot as plt
import time
from unpack_ll1 import unpack_ll1
from loewnerize_acts import loewnerize_acts
import ll1_tools
from aire import aire
from torchvision.datasets import mnist
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor, Resize, Compose
from models import Lenet5, Lenet300100


In [20]:
# Storage class for hooking into a pytorch model and collecting activation function inputs and outputs
class Storage:
    def __init__(self):
        self.storage = {}
        self.hooks = []

    def __getitem__(self, key: str):
        return self.storage[key]

    def __setitem__(self, key: str, value: torch.Tensor):
        self.storage[key] = value

    def reset(self):
        for key, val in self.storage.items():
            self.storage[key] = []

    def _from_str(self, module, text: str):
        for name in text.split("."):
            module = getattr(module, name)
        return module

    def setup(self, module = None, iter_fn: callable = "named_children", layers: List[Union[torch.nn.Module, str]] = None):
        # either input the iter func or the layers as a list of modules or strings
        def _forward(module, input, output):
            self.storage[module.mod_name].append(output)

        # if you want to see what layers we are iterating over, breakpoint here
        if layers is not None:
            layers = [(layer, self._from_str(module, layer)) if isinstance(layer, str) else layer for layer in layers]
        else:
            iter_fn = getattr(module, iter_fn) if isinstance(iter_fn, str) else iter_fn
            layers = list(iter_fn())

        name_idx = 0
        for mod in layers:
            if isinstance(mod, torch.nn.Module):
                mod_name = f"{mod.__class__.__name__}_{name_idx}"
                name_idx += 1
            else:
                mod_name, mod = mod
            # name = mod_name + '_activation'
            mod.mod_name = mod_name
            self.storage[mod_name] = []
            hook = mod.register_forward_hook(_forward)
            self.hooks.append(hook)

    def __repr__(self):
        return f"Storage: {list(self.storage.keys())}"

    def saveable(self):
        out = {}
        for key, val in self.storage.items():
            if val == []:
                continue

            out[key] = torch.stack(val).cpu().detach().numpy()
        return out


In [33]:
# Load a pre-trained model and evaluate a dataset, collecting activations for anaylsis


#********************** LENET300100 **************************
# load MNIST test data
data_dir = f'../data'
test_dataset = mnist.MNIST(
    root=f'{data_dir}',
    train=False,
    transform=ToTensor(),
    download=True
    )
# load model
model = Lenet300100(UseRational=True)
model_path = f'../data/Lenet300100/model/model_state_23.pt'
state_dict = torch.load(model_path, weights_only=True)
model.load_state_dict(state_dict)
model.to(device)

# #********************* LENET5 *****************************
# # load MNIST test data
# data_dir = f'../data'
# test_dataset = mnist.MNIST(
#     root=f'{data_dir}',
#     train=False,
#     transform=Compose([
#         viResize((32,32)),
#         ToTensor()]),
#     download=True
#     )
# # load model
# model = Lenet5(UseRational=True)
# model_path = f'../data/Lenet5/model/model_state_7.pt'
# state_dict = torch.load(model_path, weights_only=True)
# model.load_state_dict(state_dict)
# model.to(device)


batch_size = 256
test_loader = DataLoader(test_dataset, batch_size=batch_size, drop_last=True)
n_classes = 10
device = "cuda"

# set-up collection
storage = Storage()
storage.setup(model, iter_fn=model.named_modules)

# prepare to evaluate 
model.eval()
correct = 0
seen = 0

# evaluate and collect! 
storage.reset()
for idx, (test_x, test_label) in enumerate(test_loader):
    print(f'Batch: {idx}')
    test_x, test_label = test_x.to(device), test_label.to(device)
    predict_y = model(test_x.float().to(device))
    predict_ys = predict_y.argmax(dim=-1)
    correct += (predict_ys == test_label).sum().item()
    seen += len(test_label)

    # if we want loss too, we'd calculate it here
# Report accuracy
print('Accuracy: {:.4f}'.format(correct / seen))
acts = storage.saveable()

Batch: 0
Batch: 1
Batch: 2
Batch: 3
Batch: 4
Batch: 5
Batch: 6
Batch: 7
Batch: 8
Batch: 9
Batch: 10
Batch: 11
Batch: 12
Batch: 13
Batch: 14
Batch: 15
Batch: 16
Batch: 17
Batch: 18
Batch: 19
Batch: 20
Batch: 21
Batch: 22
Batch: 23
Batch: 24
Batch: 25
Batch: 26
Batch: 27
Batch: 28
Batch: 29
Batch: 30
Batch: 31
Batch: 32
Batch: 33
Batch: 34
Batch: 35
Batch: 36
Batch: 37
Batch: 38
Accuracy: 0.9799


In [53]:
# isolate labels
labels = []
for _, label in test_loader:
    labels.append(label)
labels = np.array(labels).reshape(-1)

print(f'Labels shape: {labels.shape}')

Labels shape: (9984,)


In [34]:
for key, item in acts.items():
    print(f'Key: {key}')
    print(item.shape)
    

Key: 
(39, 256, 10)
Key: layers.layer_0
(39, 256, 300)
Key: layers.layer_0.linear
(39, 256, 300)
Key: layers.layer_0.rat
(39, 256, 300)
Key: layers.layer_1
(39, 256, 100)
Key: layers.layer_1.linear
(39, 256, 100)
Key: layers.layer_1.rat
(39, 256, 100)
Key: layers.layer_2
(39, 256, 10)
Key: layers.layer_2.linear
(39, 256, 10)


In [57]:
inputs = acts['layers.layer_0.linear']
inputs = inputs.reshape(-1, inputs.shape[-1])
outputs = acts['layers.layer_0.rat']
outputs = outputs.reshape(-1, inputs.shape[-1])
print(inputs.shape)

(9984, 300)


In [59]:
# split inputs and outputs according to label
ins_0 = []
ins_1 = []
ins_2 = []
ins_3 = []
ins_4 = []
ins_5 = []
ins_6 = []
ins_7 = []
ins_8 = []
ins_9 = []
ins = [ins_0, ins_1, ins_2, ins_3, ins_4, ins_5, ins_6, ins_7, ins_8, ins_9]

outs_0 = []
outs_1 = []
outs_2 = []
outs_3 = []
outs_4 = []
outs_5 = []
outs_6 = []
outs_7 = []
outs_8 = []
outs_9 = []
outs = [outs_0, outs_1, outs_2, outs_3, outs_4, outs_5, outs_6, outs_7, outs_8, outs_9]

for idx, lbl in enumerate(labels):
    ins[lbl].append(inputs[idx])
    outs[lbl].append(outputs[idx])

for idx, inpt in enumerate(ins):
    ins[idx] = np.array(inpt)

for idx, output in enumerate(outs):
    outs[idx] = np.array(output)

print(outs[0].shape)


(979, 300)


In [35]:
print(acts['layers.layer_0'])

[[[-0.04682184 -0.03755047  0.06699296 ...  0.135299    0.95703375
   -0.22133112]
  [ 0.7118373  -0.37506714 -0.0086048  ... -0.27265963 -0.43133378
   -0.4288618 ]
  [-0.43172365 -0.09633749  0.21530843 ... -0.35164705 -0.10042272
   -0.26860908]
  ...
  [ 0.3212229   0.5444645  -0.29310387 ... -0.04042338 -0.30365646
    0.14954704]
  [-0.25766185 -0.24984512  0.04500097 ...  0.3874298   0.97133535
   -0.41635764]
  [-0.29564852 -0.40942448  0.7282459  ...  0.46298945  0.23605499
   -0.32866642]]

 [[-0.04045763 -0.29955283 -0.30921957 ... -0.39919084  0.1689621
   -0.39123583]
  [-0.42306063 -0.42553455 -0.34256044 ... -0.3912056  -0.42791858
    0.19626799]
  [-0.2796822   0.67015034  0.19643892 ...  0.43435133  0.22634469
   -0.27042145]
  ...
  [ 0.3908524   0.19579785 -0.43163043 ... -0.17149864 -0.08025608
   -0.0295199 ]
  [-0.36059943  0.00758438 -0.28762677 ... -0.16276352 -0.14237289
   -0.11689236]
  [-0.2988729  -0.27705157 -0.4100427  ... -0.0204259  -0.25847623
    0.0

In [36]:
print(acts['layers.layer_0.linear'])

[[[-0.44715834 -0.45597577  0.30030286 ...  0.33568826  1.160971
   -0.30102986]
  [ 0.7731494   0.05074173 -0.4846987  ... -0.26042888 -0.07705156
   -0.08642769]
  [-0.07519256  0.21844353  0.37913656 ... -0.1927795  -0.39904052
   -0.26367345]
  ...
  [ 0.44156945  0.6038666  -0.24384812 ...  0.24637745  0.10457617
   -0.70285934]
  [-0.27239287 -0.27858344  0.28914332 ...  0.4844988   1.1906375
   -0.11504429]
  [-0.24175505  0.01291351  0.793053   ...  0.53835535  0.3908644
   -0.21372351]]

 [[-0.45319262  0.10722853  0.10092852 ...  0.02596416 -0.74477166
   -0.1510101 ]
  [-0.10175762 -0.01546876 -0.20125017 ... -0.15104666 -0.08935927
   -0.81818235]
  [-0.2547753   0.725585    0.36864954 ...  0.5172535  -0.9413278
   -0.2622231 ]
  ...
  [ 0.48681754  0.3682961  -0.07565193 ...  0.18005374  0.22650358
   -0.46375483]
  [ 0.06330781 -0.5016762   0.11476889 ...  0.18460427  0.19511792
   -0.38504177]
  [ 0.107665   -0.2568976   0.01204561 ...  0.25634307 -0.2717464
   -0.496304

In [37]:
print(acts['layers.layer_0.rat'])


[[[-0.04682184 -0.03755047  0.06699296 ...  0.135299    0.95703375
   -0.22133112]
  [ 0.7118373  -0.37506714 -0.0086048  ... -0.27265963 -0.43133378
   -0.4288618 ]
  [-0.43172365 -0.09633749  0.21530843 ... -0.35164705 -0.10042272
   -0.26860908]
  ...
  [ 0.3212229   0.5444645  -0.29310387 ... -0.04042338 -0.30365646
    0.14954704]
  [-0.25766185 -0.24984512  0.04500097 ...  0.3874298   0.97133535
   -0.41635764]
  [-0.29564852 -0.40942448  0.7282459  ...  0.46298945  0.23605499
   -0.32866642]]

 [[-0.04045763 -0.29955283 -0.30921957 ... -0.39919084  0.1689621
   -0.39123583]
  [-0.42306063 -0.42553455 -0.34256044 ... -0.3912056  -0.42791858
    0.19626799]
  [-0.2796822   0.67015034  0.19643892 ...  0.43435133  0.22634469
   -0.27042145]
  ...
  [ 0.3908524   0.19579785 -0.43163043 ... -0.17149864 -0.08025608
   -0.0295199 ]
  [-0.36059943  0.00758438 -0.28762677 ... -0.16276352 -0.14237289
   -0.11689236]
  [-0.2988729  -0.27705157 -0.4100427  ... -0.0204259  -0.25847623
    0.0